# 01 — Data Collection: Scraping Berita AI dari Detik, CNN Indonesia, dan Liputan6

**Proyek:** Analisis Dinamika Topik Pemberitaan Generative AI pada Media Online Indonesia Menggunakan BERTopic

**Mata kuliah:** DTSC6008001 – Text Mining (Final Project)

**Video penjelasan:** `<<TEMPEL LINK VIDEO DI SINI>>`

Notebook ini menjawab Task 1. Isinya: sumber data, metode dan alat scraping, periode pengumpulan,
jumlah data, struktur kolom, pemilihan lima domain berita sebagai label kelas, dan kondisi mentah data.

## 1. Rancangan pengumpulan data

| Aspek | Keterangan |
|---|---|
| Sumber | **detik.com** (detikInet, detikFinance, detikNews, detikEdu, dll.), **cnnindonesia.com**, dan **liputan6.com** |
| Topik | Berita berbahasa Indonesia tentang AI / Generative AI |
| Periode terbit | **1 Desember 2022 – 31 Oktober 2026** (sejak ChatGPT dirilis ke publik, 30 Nov 2022) |
| Alat | Python `requests` + `BeautifulSoup` (lxml), `ThreadPoolExecutor` (4 thread), jeda 0,5 detik per permintaan |
| Kata kunci | AI, kecerdasan buatan, artificial intelligence, ChatGPT, OpenAI, AI generatif, generative AI, Gemini, Copilot, DeepSeek, deepfake, chatbot |
| Label kelas | **Domain berita** = kanal tempat artikel diterbitkan redaksi (bukan label buatan model) |

**Cara kerja per sumber**

- **Detik** — halaman pencarian `detik.com/search/searchall` mendukung filter tanggal. Scraper
  menjalankan setiap kata kunci **per bulan** (47 bulan) dan menelusuri semua halaman hasil. Pembagian
  per bulan dipakai agar tidak ada hasil yang terpotong batas halaman pencarian. Tahap ini menghasilkan
  *indeks* (URL, judul, kanal, teras, waktu terbit); isi artikel diunduh pada tahap kedua dari halaman
  artikel (`?single=1` agar artikel multi-halaman terambil utuh).
- **CNN Indonesia** — (a) API pencarian situs (`/api/v3/search`) yang langsung mengembalikan isi
  artikel, tetapi hanya menjangkau ±1 tahun terakhir; (b) **halaman tag** (`/tag/kecerdasan-buatan`,
  `/tag/chatgpt`, dst.) yang menjangkau sampai awal 2023, lalu isi artikelnya diunduh satu per satu.
- **Liputan6** — `robots.txt` melarang halaman pencarian, tetapi mengizinkan **halaman tag**
  (`/tag/kecerdasan-buatan`, `/tag/chatgpt`, `/tag/deepfake`, dst., 14 tag). Setiap tag ditelusuri sampai
  halaman terakhir (±30 halaman, menjangkau 2023–2026), lalu isi artikelnya diunduh. Kanal ada di URL
  (`liputan6.com/tekno/read/...`).

**Etika scraping.** Sebelum dipakai, setiap portal diperiksa dua hal: `robots.txt` dan pernyataan
larangan di halaman artikel/syarat penggunaan. Ketiga situs yang dipakai mengizinkan user-agent umum untuk
halaman yang diakses dan tidak mencantumkan larangan crawling. Scraper hanya mengambil halaman publik,
memberi jeda antar permintaan, dan data dipakai untuk keperluan akademik. **Antara tidak dipakai**:
walaupun `robots.txt`-nya mengizinkan, setiap artikelnya mencantumkan *"Dilarang keras mengambil konten,
melakukan crawling atau pengindeksan otomatis untuk AI ... tanpa izin tertulis dari Kantor Berita ANTARA"*.
Portal lain yang diperiksa tetapi tidak dipakai:
**Tribunnews, Kompas, Republika, IDN Times, Merdeka, Suara, Beritasatu** melarang halaman pencarian
(`/search`) dan/atau tag (`/tag`) di `robots.txt`, sedangkan API pencarian **CNBC Indonesia** hanya
menjangkau ±1 tahun terakhir sehingga akan membuat data condong ke 2026.

## 2. Kode scraper

Scraper bersifat *resumable*: hasil ditulis bertahap ke berkas JSONL dan pekerjaan yang sudah selesai
tidak diulang. Sel di bawah mendefinisikan seluruh fungsi; pemanggilannya ada di sel berikutnya.

In [ ]:
"""Scraper berita AI/Generative AI dari detik.com dan cnnindonesia.com.

Jalankan dari folder repo:
  python Group4_Scraping.py index    -> CNN (API + halaman tag) + indeks pencarian Detik
  python Group4_Scraping.py fetch    -> unduh isi artikel Detik yang ada di detik_selected.csv
                                         (detik_selected.csv dibuat oleh notebook 01)
  python Group4_Scraping.py liputan6 -> halaman tag + isi artikel Liputan6

Semua tahap resumable: hasil ditulis bertahap ke scraped/*.jsonl dan permintaan yang sudah selesai
tidak diulang. robots.txt kedua situs mengizinkan user-agent umum (`User-agent: *` / `Allow: /`);
scraper tetap memberi jeda antar permintaan.
"""
import calendar
import json
import re
import sys
import threading
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import pandas as pd
import requests
from bs4 import BeautifulSoup

ROOT = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()   # folder repo
RAW = ROOT / "scraped"
RAW.mkdir(parents=True, exist_ok=True)

UA = ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
      "(KHTML, like Gecko) Chrome/130.0 Safari/537.36")
HEADERS = {"User-Agent": UA, "Accept-Language": "id-ID,id;q=0.9,en;q=0.8"}
DELAY = 0.5          # jeda per permintaan per thread (detik)
WORKERS = int(__import__("os").environ.get("SCRAPE_WORKERS", 4))

# Periode: rilis publik ChatGPT (30 Nov 2022) s.d. 31 Okt 2026
START = (2022, 12)
END = (2026, 10)

KEYWORDS = [
    "AI", "kecerdasan buatan", "artificial intelligence", "ChatGPT", "OpenAI",
    "AI generatif", "generative AI", "Gemini", "Copilot", "DeepSeek",
    "deepfake", "chatbot",
]

_local = threading.local()
_write_lock = threading.Lock()


def session():
    if not hasattr(_local, "s"):
        _local.s = requests.Session()
        _local.s.headers.update(HEADERS)
    return _local.s


def get(url, params=None, tries=4):
    """GET dengan retry + backoff. Mengembalikan Response atau None."""
    for i in range(tries):
        try:
            r = session().get(url, params=params, timeout=30)
            if r.status_code == 200:
                time.sleep(DELAY)
                return r
            if r.status_code in (404, 410):
                return None
            time.sleep(3 * (i + 1))          # 429/5xx -> tunggu lebih lama
        except requests.RequestException:
            time.sleep(3 * (i + 1))
    return None


def append_jsonl(path, rows):
    with _write_lock, open(path, "a", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")


def read_jsonl(path):
    if not Path(path).exists():
        return []
    with open(path, encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]


def months():
    y, m = START
    while (y, m) <= END:
        yield y, m
        y, m = (y + 1, 1) if m == 12 else (y, m + 1)


# --------------------------------------------------------------------------
# DETIK: halaman pencarian per kata kunci x bulan
# --------------------------------------------------------------------------
DETIK_SEARCH = "https://www.detik.com/search/searchall"


def detik_search_page(keyword, y, m, page):
    last = calendar.monthrange(y, m)[1]
    r = get(DETIK_SEARCH, params={
        "query": keyword, "result_type": "latest",
        "fromdatex": f"01/{m:02d}/{y}", "todatex": f"{last:02d}/{m:02d}/{y}",
        "page": page,
    })
    if r is None:
        return []
    soup = BeautifulSoup(r.text, "lxml")
    rows = []
    for art in soup.select("article.list-content__item"):
        a = art.select_one("h3.media__title a") or art.select_one("a.media__link")
        if not a or not a.get("href"):
            continue
        ts = art.select_one("[d-time]")
        sub = art.select_one(".media__subtitle")
        desc = art.select_one(".media__desc")
        rows.append({
            "source": "detik",
            "url": a["href"].split("?")[0],
            "title": a.get_text(" ", strip=True),
            "channel": sub.get_text(" ", strip=True) if sub else "",
            "desc": desc.get_text(" ", strip=True) if desc else "",
            "timestamp": int(ts["d-time"]) if ts and ts.get("d-time", "").isdigit() else None,
            "keyword": keyword,
        })
    return rows


def detik_index_month(keyword, y, m, max_pages=120):
    out, page = [], 1
    while page <= max_pages:
        rows = detik_search_page(keyword, y, m, page)
        if not rows:
            break
        out.extend(rows)
        page += 1
    return out


def run_detik_index():
    path = RAW / "detik_index.jsonl"
    done_path = RAW / "detik_index_done.txt"
    done = set(done_path.read_text(encoding="utf-8").split("\n")) if done_path.exists() else set()
    jobs = [(k, y, m) for k in KEYWORDS for (y, m) in months() if f"{k}|{y}-{m:02d}" not in done]
    print(f"[detik index] {len(jobs)} job (kata kunci x bulan) tersisa", flush=True)
    n = 0
    with ThreadPoolExecutor(WORKERS) as ex:
        futs = {ex.submit(detik_index_month, *j): j for j in jobs}
        for i, fut in enumerate(as_completed(futs), 1):
            k, y, m = futs[fut]
            rows = fut.result()
            append_jsonl(path, rows)
            with _write_lock, open(done_path, "a", encoding="utf-8") as f:
                f.write(f"{k}|{y}-{m:02d}\n")
            n += len(rows)
            if i % 20 == 0 or i == len(jobs):
                print(f"  {i}/{len(jobs)} job, +{n} baris", flush=True)


# --------------------------------------------------------------------------
# CNN INDONESIA: API pencarian (mengembalikan isi artikel lengkap)
# --------------------------------------------------------------------------
CNN_API = "https://www.cnnindonesia.com/api/v3/search"


def html_to_text(html):
    soup = BeautifulSoup(html or "", "lxml")
    for t in soup(["script", "style", "table", "iframe", "figure"]):
        t.decompose()
    paras = [p.get_text(" ", strip=True) for p in soup.find_all("p")]
    paras = [p for p in paras if p]
    return "\n".join(paras) if paras else soup.get_text(" ", strip=True)


def run_cnn():
    path = RAW / "cnn_articles.jsonl"
    seen = {r["url"] for r in read_jsonl(path)}
    for kw in KEYWORDS:
        start, got = 0, 0
        while True:
            r = get(CNN_API, params={"query": kw, "idtype": 1, "start": start,
                                     "limit": 50, "pageType": "search"})
            data = (r.json().get("data") or []) if r is not None else []
            if not data:
                break
            rows = []
            for d in data:
                url = d.get("url") or ""
                if not url or url in seen:
                    continue
                seen.add(url)
                rows.append({
                    "source": "cnn",
                    "url": url,
                    "title": d.get("strjudul", ""),
                    "channel": url.split("/")[3] if url.count("/") > 3 else "",
                    "subchannel": d.get("strnmkanal", ""),
                    "desc": d.get("strringkasan", ""),
                    "date": (d.get("dtnewsdate") or "").replace("/", "-"),
                    "tags": d.get("strtag", ""),
                    "body": html_to_text(d.get("strisi", "")),
                    "keyword": kw,
                })
            append_jsonl(path, rows)
            got += len(rows)
            start += 50
        print(f"[cnn] '{kw}': +{got} artikel baru (total {len(seen)})", flush=True)


# --------------------------------------------------------------------------
# CNN INDONESIA: halaman tag (API pencarian hanya menjangkau +-1 tahun terakhir,
# halaman tag menjangkau sampai awal 2023) + isi artikel
# --------------------------------------------------------------------------
CNN_TAGS = ["kecerdasan-buatan", "ai", "artificial-intelligence", "chatgpt", "openai", "chatbot",
            "deepfake", "gemini", "deepseek", "teknologi-ai", "generative-ai", "ai-generatif",
            "copilot", "google-gemini", "sam-altman", "nvidia"]
CNN_URL_RE = re.compile(r"cnnindonesia\.com/([a-z-]+)/(20\d{12})-\d+-\d+/")


def cnn_tag_pages(tag, max_pages=400):
    out, page, empty = [], 1, 0
    while page <= max_pages and empty < 2:
        r = get(f"https://www.cnnindonesia.com/tag/{tag}", params={"page": page})
        rows = []
        if r is not None:
            soup = BeautifulSoup(r.text, "lxml")
            for a in soup.select("article a[href]"):
                m = CNN_URL_RE.search(a["href"])
                h = a.select_one("h2")
                if m and h:
                    rows.append({"source": "cnn", "url": a["href"].split("?")[0],
                                 "title": h.get_text(" ", strip=True), "channel": m.group(1), "tag": tag})
        empty = empty + 1 if not rows else 0
        out.extend(rows)
        page += 1
    return out


def run_cnn_tags():
    path = RAW / "cnn_tag_index.jsonl"
    done = {r["tag"] for r in read_jsonl(path)}
    with ThreadPoolExecutor(WORKERS) as ex:
        futs = {ex.submit(cnn_tag_pages, t): t for t in CNN_TAGS if t not in done}
        for fut in as_completed(futs):
            rows = fut.result()
            append_jsonl(path, rows)
            print(f"[cnn tag] {futs[fut]}: {len(rows)} tautan", flush=True)


def cnn_article(row):
    r = get(row["url"])
    if r is None:
        return {**row, "ok": False}
    soup = BeautifulSoup(r.text, "lxml")
    body = soup.select_one(".detail-text")
    if body is None:
        return {**row, "ok": False}
    for t in body.select("script, style, table, iframe, figure, .paradetail, .para_caption"):
        t.decompose()
    paras = [p.get_text(" ", strip=True) for p in body.find_all("p")]

    def meta(name):
        t = soup.find("meta", attrs={"name": name})
        return t.get("content", "") if t else ""

    return {**row, "ok": True, "desc": meta("description"),
            "date": meta("publishdate").replace("/", "-"), "tags": meta("keywords"),
            "subchannel": meta("dtk:namakanal"), "body": "\n".join(p for p in paras if p), "keyword": "tag:" + row["tag"]}


def run_cnn_fetch():
    path = RAW / "cnn_articles.jsonl"
    have = {r["url"] for r in read_jsonl(path)}
    todo = {}
    for r in read_jsonl(RAW / "cnn_tag_index.jsonl"):
        if r["url"] not in have and "/tv/" not in r["url"]:
            todo.setdefault(r["url"], r)
    print(f"[cnn fetch] {len(todo)} artikel dari halaman tag belum ada", flush=True)
    with ThreadPoolExecutor(WORKERS) as ex:
        futs = [ex.submit(cnn_article, r) for r in todo.values()]
        for i, fut in enumerate(as_completed(futs), 1):
            res = fut.result()
            if res.get("ok"):
                res.pop("ok"); res.pop("tag", None)
                append_jsonl(path, [res])
            if i % 250 == 0 or i == len(futs):
                print(f"  {i}/{len(futs)}", flush=True)


# --------------------------------------------------------------------------
# DETIK: isi artikel
# --------------------------------------------------------------------------
def detik_article(url):
    r = get(url, params={"single": 1})
    if r is None:
        return {"url": url, "ok": False}
    soup = BeautifulSoup(r.text, "lxml")
    body = soup.select_one(".detail__body-text") or soup.select_one(".itp_bodycontent")
    if body is None:
        return {"url": url, "ok": False}
    for t in body.select("script, style, table, iframe, .aevp, .embed, .detail__body-tag, "
                         ".ads-scrollpage-container, .lihatjg, .sisip_embed_sosmed, .staticdetail_container"):
        t.decompose()
    paras = [p.get_text(" ", strip=True) for p in body.find_all("p")]
    paras = [p for p in paras if p]
    h1 = soup.select_one("h1")

    def meta(name):
        t = soup.find("meta", attrs={"name": name})
        return t.get("content", "") if t else ""

    return {
        "url": url, "ok": True,
        "title_page": h1.get_text(" ", strip=True) if h1 else "",
        "date": meta("publishdate").replace("/", "-"),
        "kanalid": meta("kanalid"),
        "author": meta("author"),
        "tags": meta("keywords"),
        "body": "\n".join(paras),
    }


def run_detik_fetch():
    sel = pd.read_csv(ROOT / "detik_selected.csv")
    path = RAW / "detik_articles.jsonl"
    done = {r["url"] for r in read_jsonl(path)}
    urls = [u for u in sel["url"].tolist() if u not in done]
    print(f"[detik fetch] {len(urls)} artikel tersisa dari {len(sel)}", flush=True)
    with ThreadPoolExecutor(WORKERS) as ex:
        futs = [ex.submit(detik_article, u) for u in urls]
        for i, fut in enumerate(as_completed(futs), 1):
            append_jsonl(path, [fut.result()])
            if i % 250 == 0 or i == len(urls):
                print(f"  {i}/{len(urls)}", flush=True)


# --------------------------------------------------------------------------
# LIPUTAN6: halaman tag (robots.txt mengizinkan /tag/, melarang /search) + isi artikel.
# Catatan: Antara tidak dipakai karena setiap artikelnya mencantumkan larangan crawling/pengindeksan
# otomatis untuk AI tanpa izin tertulis, walaupun robots.txt-nya mengizinkan.
# --------------------------------------------------------------------------
L6_TAGS = ["kecerdasan-buatan", "artificial-intelligence", "ai", "chatgpt", "openai", "deepfake", "chatbot",
           "generative-ai", "ai-generatif", "gemini-ai", "deepseek", "teknologi-ai", "google-gemini", "copilot"]
L6_URL_RE = re.compile(r"https://www\.liputan6\.com/([a-z0-9-]+)/read/(\d+)/[^?#\"]+")


def l6_tag_pages(tag, max_pages=60):
    out, page = [], 1
    while page <= max_pages:
        r = get(f"https://www.liputan6.com/tag/{tag}", params={"page": page})
        if r is None:                       # 404 = melewati halaman terakhir
            break
        soup = BeautifulSoup(r.content, "lxml")
        main = soup.select_one("div.tags--layout--right_all")
        rows = []
        for item in (main.select("article.articles--iridescent-list--item") if main else []):
            a = item.select_one("a[href*='/read/']")
            m = L6_URL_RE.search(a["href"]) if a else None
            if not m:
                continue
            t = item.select_one("time[datetime]")
            title = item.select_one("[class*=title]")
            rows.append({"source": "liputan6", "url": m.group(0), "channel": m.group(1), "tag": tag,
                         "title": (title or a).get_text(" ", strip=True),
                         "date": t["datetime"][:19].replace("T", " ") if t else ""})
        if not rows:
            break
        out.extend(rows)
        page += 1
    return out


def run_l6_index():
    path = RAW / "liputan6_index.jsonl"
    done = {r["tag"] for r in read_jsonl(path)}
    with ThreadPoolExecutor(WORKERS) as ex:
        futs = {ex.submit(l6_tag_pages, t): t for t in L6_TAGS if t not in done}
        for fut in as_completed(futs):
            rows = fut.result()
            append_jsonl(path, rows)
            print(f"[liputan6 tag] {futs[fut]}: {len(rows)} tautan", flush=True)


def l6_article(row):
    r = get(row["url"])
    if r is None:
        return {**row, "ok": False}
    soup = BeautifulSoup(r.content, "lxml")
    blocks = soup.select(".article-content-body__item-content")
    if not blocks:
        return {**row, "ok": False}
    paras = []
    for blk in blocks:
        for t in blk.select("script, style, table, iframe, figure, .baca-juga, .advertisement"):
            t.decompose()
        paras += [p.get_text(" ", strip=True) for p in blk.find_all("p")]

    def meta(name):
        t = soup.find("meta", attrs={"name": name}) or soup.find("meta", attrs={"property": name})
        return t.get("content", "") if t else ""

    return {**row, "ok": True, "desc": meta("description"), "tags": meta("keywords"),
            "date": meta("article:published_time")[:19].replace("T", " ") or row.get("date", ""),
            "body": "\n".join(p for p in paras if p), "keyword": "tag:" + row["tag"]}


def run_l6_fetch():
    todo = {}
    for r in read_jsonl(RAW / "liputan6_index.jsonl"):
        todo.setdefault(r["url"], r)
    path = RAW / "liputan6_articles.jsonl"
    have = {r["url"] for r in read_jsonl(path)}
    rows = [r for u, r in todo.items() if u not in have]
    print(f"[liputan6 fetch] {len(rows)} artikel tersisa dari {len(todo)}", flush=True)
    with ThreadPoolExecutor(WORKERS) as ex:
        futs = [ex.submit(l6_article, r) for r in rows]
        for i, fut in enumerate(as_completed(futs), 1):
            res = fut.result()
            res.pop("tag", None)
            append_jsonl(path, [res])
            if i % 250 == 0 or i == len(rows):
                print(f"  {i}/{len(rows)}", flush=True)


if __name__ == "__main__":
    step = sys.argv[1] if len(sys.argv) > 1 else "index"
    if step == "index":
        run_cnn()
        run_cnn_tags()
        run_cnn_fetch()
        run_detik_index()
    elif step == "fetch":
        run_detik_fetch()
    elif step == "liputan6":
        run_l6_index()
        run_l6_fetch()

Scraping penuh memakan waktu beberapa jam. `RUN_SCRAPING = False` membuat notebook memakai hasil yang
sudah tersimpan di folder `scraped/`; ubah menjadi `True` untuk mengulang dari awal (yang sudah ada dilewati).

In [ ]:
RUN_SCRAPING = False
if RUN_SCRAPING:
    run_cnn()            # CNN: API pencarian
    run_cnn_tags()       # CNN: indeks halaman tag
    run_cnn_fetch()      # CNN: isi artikel dari halaman tag
    run_detik_index()    # Detik: indeks pencarian per kata kunci x bulan
    run_l6_index()       # Liputan6: indeks halaman tag
    run_l6_fetch()       # Liputan6: isi artikel

### Fungsi bantu: pemetaan kanal → domain dan kriteria relevansi AI

`domain_of(url)` memetakan kanal penerbit (dari URL) ke domain berita untuk ketiga portal. `ai_mentions` dan `is_relevant`
adalah aturan relevansi AI yang juga dipakai saat cleaning di notebook 02.

In [ ]:
"""Pemetaan kanal portal -> domain berita (label kelas)."""
import re

DETIK_HOST = {
    "inet.detik.com": "Teknologi",
    "finance.detik.com": "Ekonomi",
    "health.detik.com": "Kesehatan",
    "hot.detik.com": "Hiburan",
    "wolipop.detik.com": "Hiburan",
    "oto.detik.com": "Otomotif",
    "sport.detik.com": "Olahraga",
    "travel.detik.com": "Travel",
    "food.detik.com": "Kuliner",
}
DETIK_WWW = {"edu": "Pendidikan", "pop": "Hiburan", "hikmah": "Religi", "properti": "Properti"}
DETIK_REGIONAL = {"jabar", "jatim", "jateng", "jogja", "sulsel", "sumut", "bali", "sumbagsel", "kalimantan", "sumbar"}
REGIONAL_SUB = {"bisnis": "Ekonomi", "berita": "Nasional", "hukum-dan-kriminal": "Nasional", "pemilu": "Nasional"}
CNN_CHANNEL = {
    "teknologi": "Teknologi", "ekonomi": "Ekonomi", "nasional": "Nasional",
    "internasional": "Internasional", "edukasi": "Pendidikan", "hiburan": "Hiburan",
    "gaya-hidup": "Hiburan", "olahraga": "Olahraga", "otomotif": "Otomotif",
}
SKIP = re.compile(r"20\.detik\.com|/foto|/video|/infografis|/detiktv|/tv/|/d-\d+/$")


# Liputan6: kanal ada di URL (liputan6.com/<kanal>/read/...). Kanal di luar lima domain (global, otomotif,
# info-sehat, cek-fakta, opini, citizen6, dll.) tidak dipakai.
L6_CHANNEL = {"tekno": "Teknologi", "bisnis": "Ekonomi", "saham": "Ekonomi", "crypto": "Ekonomi",
              "news": "Nasional", "politik": "Nasional", "peristiwa": "Nasional", "regional": "Nasional",
              "showbiz": "Hiburan", "hot": "Hiburan", "lifestyle": "Hiburan"}


def domain_of(url):
    """Kembalikan nama domain berita, atau None bila bukan artikel teks / di luar pemetaan."""
    if SKIP.search(url):
        return None
    m = re.match(r"https?://([^/]+)/([^/]+)/?([^/]*)", url)
    if not m:
        return None
    host, p1, p2 = m.groups()
    if host == "www.cnnindonesia.com":
        return CNN_CHANNEL.get(p1)
    if host == "www.liputan6.com":
        return L6_CHANNEL.get(p1) if p2 == "read" else None
    if host in DETIK_HOST:
        return DETIK_HOST[host]
    if host == "news.detik.com":
        if p1 in ("internasional", "dw", "bbc-world", "abc-australia", "bbc"):
            return "Internasional"
        return "Nasional" if p1 in ("berita", "pemilu", "pilkada", "berita-jawa-barat", "berita-jawa-timur",
                                    "berita-jawa-tengah", "daerah") else None
    if host == "www.detik.com":
        if p1 in DETIK_WWW:
            return DETIK_WWW[p1]
        if p1 in DETIK_REGIONAL:
            return REGIONAL_SUB.get(p2)
    return None

In [ ]:
"""Kriteria relevansi artikel terhadap topik AI / Generative AI.

Artikel dianggap relevan bila AI adalah POKOK berita, bukan sekadar disebut sepintas:
  - judul menyebut istilah AI, ATAU
  - paragraf pembuka (3 paragraf pertama) menyebut istilah AI DAN seluruh isi menyebut istilah AI >= 3 kali.
Berita memakai struktur piramida terbalik: bila AI tidak muncul di judul maupun pembuka, AI hampir pasti
bukan pokok beritanya.

Kata "AI" sendirian ambigu di berita Indonesia:
  - inisial orang: "tersangka AI (25)", "berinisial AI", "Bripka AI", "yakni AI dan A";
  - singkatan lain: "avian influenza (AI)" (flu burung), Amnesty International;
  - "copilot" juga berarti kopilot pesawat ("pilot dan copilot"), yang tidak dihitung.
Pada teks yang memakai "AI" dengan arti lain, kata "AI" tidak dihitung sama sekali.
"""
import re

AI_TERMS = (
    r"kecerdasan buatan|kecerdasan artifisial|artificial intelligence|akal imitasi|"
    r"generative ai|ai generatif|genai|chatgpt|chat gpt|gpt-?\d[\w.]*|openai|gemini ai|google gemini|"
    r"(?<!pilot-)(?<!pilot dan )(?<!pilot )copilot|"
    r"deepseek|deepfake|chatbot|midjourney|large language model|llm|machine learning|"
    r"meta ai|grok ai|claude ai|anthropic"
)
TERM_PATTERN = re.compile(rf"\b(?:{AI_TERMS})\b", re.IGNORECASE)
AI_WORD = re.compile(r"\bA\.?I\b")                       # "AI" huruf kapital

# Penanda orang/pangkat sebelum inisial. "korban"/"pelaku" sengaja tidak dipakai karena berita AI
# sering menulis "korban AI" (korban penyalahgunaan AI) atau "pelaku AI" (pelaku industri AI).
PERSON_CUE = (
    r"inisial|berinisial|berinisal|inisal|tersangka|terdakwa|terpidana|saksi|bocah|sopir|"
    r"sdr\.?|sdri\.?|tn\.?|ny\.?|"
    r"bripka|briptu|bripda|brigadir|aipda|aiptu|ipda|iptu|akp|kompol|serda|sertu|serka|kopda|koptu|pratu|prada"
)
INITIAL_PATTERN = re.compile(
    rf"(?i:\b(?:{PERSON_CUE}))\s+(?:[A-Z]{{1,4}}\s*(?:,|dan|&)\s*)*A\.?I\b"     # "berinisial (RS dan) AI"
    r"|\bA\.?I\s*\(\s*\d{1,2}\s*(?:tahun|th)?\s*\)"                          # "AI (25)"
    r"|\b(?:yakni|yaitu)\s+A\.?I\s+dan\s+[A-Z]{1,3}\b"                        # "yakni AI dan A"
    r"|\badalah\s+A\.?I\s*,\s*(?:seorang|warga|pria|wanita|pemilik|sopir)"   # "adalah AI, seorang ..."
    # relasi keluarga + inisial: "keponakannya, AI", "anaknya AI"
    r"|(?i:\b(?:keponakan|anak|adik|kakak|istri|suami|ayah|cucu|menantu|mertua)(?:nya)?)\s*,?\s+A\.?I\b"
    # dua inisial orang: "AI dan AJ" (bukan akronim teknologi seperti "AI dan ML")
    r"|\bA\.?I\s+dan\s+(?!(?:ML|AR|VR|XR|BI|IT|TI|PC|UI|UX|HP|EV|DL|AI)\b)[A-Z]{1,2}\b"
)
OTHER_SENSE = re.compile(r"avian influenza|flu burung|amnesty international", re.IGNORECASE)

# pola gabungan (dipakai untuk penandaan cepat pada judul/teras)
AI_PATTERN = re.compile(TERM_PATTERN.pattern + r"|\bA\.?I\b")


def ai_mentions(text, gap=25):
    """Jumlah penyebutan AI tanpa hitung ganda.

    - Istilah AI dan kata "AI" yang berjarak < `gap` karakter digabung menjadi satu penyebutan,
      sehingga "kecerdasan buatan atau artificial intelligence (AI)" = 1 penyebutan.
    - Bila teks memakai "AI" sebagai inisial orang atau dengan arti lain, kata "AI" di teks itu tidak
      dihitung; hanya istilah tegas (kecerdasan buatan, ChatGPT, deepfake, ...).
    """
    text = text or ""
    pos = [(m.start(), m.end()) for m in TERM_PATTERN.finditer(text)]
    if not INITIAL_PATTERN.search(text) and not OTHER_SENSE.search(text):
        pos += [(m.start(), m.end()) for m in AI_WORD.finditer(text)]
    pos.sort()
    n, last_end = 0, -10 ** 9
    for a, b in pos:
        if a - last_end >= gap:
            n += 1
        last_end = max(last_end, b)
    return n


def lead_text(body, n_par=3):
    return "\n".join([p for p in (body or "").split("\n") if p.strip()][:n_par])


def is_relevant(title, body):
    """AI di judul, ATAU AI di paragraf pembuka DAN >= 3 penyebutan AI di seluruh isi.
    Judul dinilai bersama isinya agar aturan inisial/arti lain di isi juga berlaku untuk judul."""
    full = f"{title}\n{body}"
    if ai_mentions(title) >= 1 and ai_mentions(full) >= 1:
        return True
    return ai_mentions(lead_text(body)) >= 1 and ai_mentions(full) >= 3

## 3. Indeks hasil pencarian Detik

In [ ]:
idx = pd.DataFrame(read_jsonl(RAW / "detik_index.jsonl"))
print("baris indeks (masih ada duplikat antar kata kunci):", len(idx))
kw_hits = idx.groupby("keyword")["url"].nunique().sort_values(ascending=False)
idx = idx.drop_duplicates("url").reset_index(drop=True)
idx["date"] = pd.to_datetime(idx["timestamp"], unit="s", utc=True).dt.tz_convert("Asia/Jakarta").dt.tz_localize(None)
idx["domain"] = idx["url"].map(domain_of)
print("URL unik:", len(idx), "| rentang tanggal:", idx.date.min().date(), "s.d.", idx.date.max().date())
kw_hits.rename("URL unik per kata kunci").to_frame()

Pencarian kata "AI" di Detik juga mengembalikan artikel yang tidak membahas kecerdasan buatan (mis.
nama orang atau kata daerah). Karena itu tiap URL diberi penanda `ai_hit`: judul atau teras memuat
istilah AI (fungsi `ai_mentions`). Artikel video/foto dan kanal di luar pemetaan
domain dibuang. Penanda ini dipakai pada pemilihan URL di bagian 4; penyaringan relevansi yang
sebenarnya (berdasarkan **isi** artikel) dilakukan di notebook 02.

In [ ]:
idx["ai_hit"] = (idx["title"] + " " + idx["desc"]).map(ai_mentions) > 0
cand = idx[idx.ai_hit & idx.domain.notna()].copy()
print(f"judul/teras menyebut AI dan domain terpetakan: {len(cand)} dari {len(idx)} URL unik")
pd.crosstab(idx["domain"].fillna("(bukan artikel teks / di luar pemetaan)"), idx["ai_hit"]).sort_values(True, ascending=False)

**Interpretasi indeks pencarian.** Dari 47.029 baris hasil pencarian Detik diperoleh 27.524 URL unik.
Kata kunci "AI" menyumbang URL terbanyak (21.177) tetapi juga paling "kotor": banyak hasilnya tidak
membahas kecerdasan buatan. Sebaliknya, kata kunci spesifik (ChatGPT, DeepSeek, deepfake) sedikit
tetapi hampir selalu relevan. Kata kunci "Gemini" bahkan menarik ±3.500 artikel **ramalan zodiak** —
contoh nyata mengapa hasil scraping tidak boleh langsung dipakai tanpa penyaringan relevansi.

## 4. Pemilihan lima domain (label kelas)

Soal meminta lima domain berita sebagai label. Kandidat domain dari kedua sumber digabung, lalu dipilih
lima domain dengan jumlah artikel terbanyak yang **tersedia di kedua sumber**.

In [ ]:
cnn = pd.DataFrame(read_jsonl(RAW / "cnn_articles.jsonl")).drop_duplicates("url")
cnn["domain"] = cnn["url"].map(domain_of)
cnn["ai_hit"] = (cnn["title"] + " " + cnn["desc"].fillna("") + " " + cnn["body"].fillna("")).map(ai_mentions) > 0
dom = pd.concat([cand.groupby("domain").size().rename("detik"),
                 cnn[cnn.ai_hit].groupby("domain").size().rename("cnn")], axis=1).fillna(0).astype(int)
dom["total"] = dom.sum(1)
dom = dom.sort_values("total", ascending=False)
dom

In [ ]:
# Lima domain terpilih. "Hiburan" mencakup hiburan & gaya hidup
# (detikHot, detikPop, Wolipop, CNN Hiburan, CNN Gaya Hidup).
CLASSES = ["Teknologi", "Nasional", "Ekonomi", "Pendidikan", "Hiburan"]

# Teknologi sudah sangat banyak -> cukup artikel yang judul/terasnya menyebut AI.
# Empat domain lain jauh lebih kecil -> SEMUA hasil pencarian diunduh, lalu relevansinya
# dinilai dari isi artikel di notebook 02 (aturan di src/relevance.py).
selected = idx[idx.domain.isin(CLASSES) & (idx.ai_hit | (idx.domain != "Teknologi"))]
selected = selected[["url", "title", "desc", "channel", "domain", "keyword", "date"]].reset_index(drop=True)
selected["domain"].value_counts()

**Alasan memilih lima domain ini.** Teknologi, Nasional, Ekonomi, Pendidikan, dan Hiburan adalah lima
domain dengan artikel AI terbanyak di Detik dan CNN. Domain lain (Internasional, Otomotif, Kesehatan,
dst.) terlalu kecil untuk dilatih dan diringkas per topik. Hiburan digabung dengan Gaya Hidup karena di
Detik keduanya berada di kanal yang sama (detikHot, detikPop, Wolipop).

Untuk Teknologi cukup diunduh artikel yang judul/terasnya menyebut AI (jumlahnya sudah ribuan). Untuk
empat domain lain **semua** hasil pencarian diunduh, karena di kanal non-teknologi AI sering hanya muncul
di isi berita; relevansinya baru dinilai dari isi artikel di notebook 02.

## 5. Mengunduh isi artikel Detik

Daftar URL terpilih disimpan ke `detik_selected.csv`, lalu `run_detik_fetch()` mengunduh isinya.

In [ ]:
sel_path = ROOT / "detik_selected.csv"
if not sel_path.exists():
    selected.to_csv(sel_path, index=False)
selected = pd.read_csv(sel_path)
if RUN_SCRAPING:
    run_detik_fetch()
print("URL Detik terpilih:", len(selected))
selected["domain"].value_counts()

## 6. Artikel Liputan6

Semua artikel dari halaman tag diunduh isinya, lalu kanal (dari URL) dipetakan ke domain.

In [ ]:
l6 = pd.DataFrame(read_jsonl(RAW / "liputan6_articles.jsonl")).drop_duplicates("url")
print("Liputan6: URL unik dari 14 tag", len(set(r["url"] for r in read_jsonl(RAW / "liputan6_index.jsonl"))),
      "| terunduh", int(l6.ok.sum()), "| gagal", int((~l6.ok).sum()))
l6 = l6[l6.ok].copy()
l6["domain"] = l6["url"].map(domain_of)
l6["source"] = "liputan6"
l6.groupby("channel").size().rename("artikel").to_frame().join(
    l6.groupby("channel")["domain"].first().fillna("— dibuang —")).sort_values("artikel", ascending=False)

**Interpretasi Liputan6.** Dari 2.285 URL di 14 halaman tag, 2.283 berhasil diunduh. Kanal terbanyak
adalah Tekno (1.322), lalu Bisnis/Crypto/Saham (Ekonomi) dan Hot/Lifestyle/Showbiz (Hiburan). Kanal di
luar lima domain (Cek Fakta, Global, Health, Otomotif, dll.) dibuang. Liputan6 terutama menambah artikel
**Hiburan** dan **Ekonomi**, dua kelas yang paling kecil di Detik dan CNN.

## 7. Dataset mentah gabungan

In [ ]:
det = pd.DataFrame(read_jsonl(RAW / "detik_articles.jsonl")).drop_duplicates("url")
print("Detik: diminta", len(selected), "| terunduh", len(det), "| gagal/tanpa isi", int((~det.ok).sum()))
det = det[det.ok].merge(selected[["url", "title", "desc", "channel", "domain", "keyword"]], on="url", how="inner")
det["source"] = "detik"
cnn_keep = cnn[cnn.domain.isin(CLASSES)].copy()
cols = ["source", "url", "domain", "channel", "title", "desc", "date", "tags", "body", "keyword"]
l6_keep = l6[l6.domain.isin(CLASSES)].copy()
raw = pd.concat([det[cols], cnn_keep[cols], l6_keep[cols]], ignore_index=True)
raw["date"] = pd.to_datetime(raw["date"], errors="coerce")
raw.to_csv(ROOT / "raw_dataset.csv", index=False, encoding="utf-8-sig")   # BOM: aman dibuka di Excel
print("Total record mentah:", len(raw))
print("Rentang tanggal terbit:", raw.date.min(), "s.d.", raw.date.max())
pd.crosstab(raw["domain"], raw["source"], margins=True)

### Struktur kolom

| Kolom | Isi |
|---|---|
| `source` | portal asal (`detik` / `cnn` / `liputan6`) |
| `url` | alamat artikel (kunci unik) |
| `domain` | domain berita = **label kelas** (dari kanal penerbit) |
| `channel` | nama kanal asli di portal |
| `title` | judul artikel |
| `desc` | teras/deskripsi singkat yang ditulis redaksi (dipakai sebagai ringkasan referensi di Task 5) |
| `date` | waktu terbit |
| `tags` | kata kunci/tag dari redaksi |
| `body` | **teks utama** artikel (paragraf digabung dengan baris baru) |
| `keyword` | kata kunci/tag pencarian yang pertama kali menemukan artikel |

In [ ]:
pd.set_option("display.max_colwidth", 90)
raw.sample(5, random_state=7)[["source", "domain", "date", "title", "body"]]

## 8. Kondisi mentah data (kualitatif)

Pemeriksaan cepat terhadap teks mentah untuk menentukan langkah pembersihan di notebook 02.

In [ ]:
import re
checks = {
    "memuat 'SCROLL TO CONTINUE WITH CONTENT'": raw.body.str.contains("SCROLL TO CONTINUE", case=False),
    "memuat 'ADVERTISEMENT'": raw.body.str.contains("ADVERTISEMENT"),
    "memuat 'Baca juga' / 'Simak juga' / 'Lihat juga'": raw.body.str.contains(r"(?:Baca|Simak|Lihat) juga", case=False),
    "memuat '[Gambas:...]' (sisa embed CNN)": raw.body.str.contains(r"\[Gambas:"),
    "memuat nama media (detik*/CNN Indonesia/Liputan6) di isi": raw.body.str.contains(r"detik[a-zA-Z]+|CNN Indonesia|Liputan6"),
    "memuat URL": raw.body.str.contains(r"https?://"),
    "isi < 100 kata": raw.body.str.split().str.len() < 100,
    "judul duplikat": raw.title.str.lower().duplicated(keep=False),
    "tanggal di luar periode": ~raw.date.between("2022-12-01", "2026-10-31 23:59:59"),
}
pd.DataFrame({"jumlah": {k: int(v.sum()) for k, v in checks.items()},
              "persen": {k: round(100 * v.mean(), 1) for k, v in checks.items()}})

In [ ]:
ex = raw[raw.body.str.contains("SCROLL TO CONTINUE", case=False)].iloc[0]
print(ex.url, "\n")
print(ex.body[:1200])

**Temuan kondisi mentah**

- Teks berita relatif baku (ditulis jurnalis), tidak seperti media sosial — hampir tidak ada bahasa gaul.
- Namun ada **boilerplate situs** yang ikut terambil: penanda iklan (`SCROLL TO CONTINUE WITH CONTENT`,
  `ADVERTISEMENT`), tautan silang (`Baca juga: …`), sisa embed (`[Gambas:Video CNN]`), dan inisial redaksi
  di akhir artikel. Artikel Liputan6 diawali *dateline* `Liputan6.com, Jakarta -`.
- **Nama media dan kanal muncul di dalam isi** (`detikINET`, `detikcom`, `CNN Indonesia`, `Liputan6.com`). Karena label
  kelas berasal dari kanal, kata-kata ini akan **membocorkan label** ke model klasifikasi dan harus dibuang.
- Ada artikel yang sama ditemukan lewat beberapa kata kunci (sudah dideduplikasi lewat URL) dan sejumlah
  judul kembar/hampir kembar (berita yang diterbitkan ulang di kanal lain).
- Sebagian hasil pencarian tidak benar-benar membahas AI; penyaringan relevansi dilakukan di notebook 02.

## Ringkasan Task 1

| Aspek | Hasil |
|---|---|
| Sumber | detik.com, cnnindonesia.com, liputan6.com (robots.txt dan syarat konten diperiksa; Antara ditolak karena melarang crawling untuk AI) |
| Metode & alat | `requests` + `BeautifulSoup`; Detik lewat halaman pencarian per kata kunci × bulan, CNN lewat API pencarian + halaman tag, Liputan6 lewat halaman tag |
| Periode | artikel terbit 1 Desember 2022 – 31 Oktober 2026; scraping dilakukan November 2026 |
| Jumlah record mentah | **23.024 artikel** di 5 domain → setelah cleaning dan penyaringan relevansi AI di notebook 02 menjadi **6.714 artikel** (≥ 5.000) |
| Label | 5 domain berita dari kanal penerbit: Teknologi, Nasional, Ekonomi, Pendidikan, Hiburan |
| Kolom | `source, url, domain, channel, title, desc, date, tags, body, keyword` (kolom `url` = tautan sumber) |